<a href="https://colab.research.google.com/github/Ankitp2002/RAG_with_milvus/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Wrap the brackets in quotes so Colab's bash passes them accurately
!pip install 'markitdown[pdf]' 'markitdown[all]'

# Directly force-install the underlying engines used by MarkItDown for PDFs
!pip install pdfplumber pdfminer.six pymupdf

# Install the core native SDKs and the local Milvus Lite engine
!pip install -U pymilvus milvus-lite markitdown groq sentence-transformers

# Install essential Linux binaries for parsing heavy formats and layout assets in Colab
!apt-get update && apt-get install -y poppler-utils tesseract-ocr

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the ne

In [2]:
from google.colab import userdata

In [6]:
import os
from markitdown import MarkItDown
from groq import Groq
from pymilvus import MilvusClient, DataType
from sentence_transformers import SentenceTransformer

# =====================================================================
# 1. SETUP & COLAB LOCAL STORAGE ENVIRONMENT
# =====================================================================

# Model configurations
VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"
DIMENSION = 1024  # Standard dense array dimension output for BGE-Large
COLLECTION_NAME = "colab_local_rag_collection"

# Absolute local file path inside the Google Colab environment space
DB_FILE_PATH = os.path.abspath("/content/local_milvus_storage.db")

# Ensure Groq API Key is present in the Colab session environment variables
if not userdata.get('GROQ_API_KEY'):
    # If not set in Colab secrets, you can manually set it here:
    # os.environ["GROQ_API_KEY"] = "gsk_..."
    print("⚠️ WARNING: Please ensure GROQ_API_KEY is configured in your environment or Colab secrets.")

# Initialize native infrastructure clients
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))
embedding_model = SentenceTransformer(EMBED_MODEL_NAME)

# Passing a local file string automatically instantiates Milvus Lite locally
milvus_client = MilvusClient(uri=DB_FILE_PATH)

# Build the structural database schema natively
if not milvus_client.has_collection(COLLECTION_NAME):
    schema = milvus_client.create_schema(auto_id=True, enable_dynamic_field=True)
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=DIMENSION)
    schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)

    index_params = milvus_client.prepare_index_params()
    index_params.add_index(field_name="vector", metric_type="COSINE", index_type="AUTOINDEX")

    milvus_client.create_collection(
        collection_name=COLLECTION_NAME,
        schema=schema,
        index_params=index_params
    )
    print(f"[✓] Milvus Lite database file successfully initialized at: {DB_FILE_PATH}")

# Bind MarkItDown natively to utilize Llama-4-Scout for visual components
md_parser = MarkItDown(llm_client=groq_client, llm_model=VISION_MODEL)

# =====================================================================
# 2. CORE INSERTION PIPELINE (MANUAL INGEST)
# =====================================================================

def ingest_unstructured_document(file_path: str):
    """
    Parses document structures locally (tables convert to markdown).
    Images are securely translated via Llama-4-Scout over Groq.
    Generates local vector arrays and deposits them straight into the local .db file.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Provided path does not exist in Colab workspace: {file_path}")

    print(f"[*] Parsing file via MarkItDown: {file_path}")
    conversion = md_parser.convert(file_path)
    markdown_content = conversion.text_content

    # Split text cleanly by logical markdown segment boundaries
    text_chunks = [chunk.strip() for chunk in markdown_content.split("\n\n") if chunk.strip()]

    payload = []
    print(f"[*] Transforming {len(text_chunks)} segments into vectors using {EMBED_MODEL_NAME}...")
    for chunk in text_chunks:
        # Local transformation calculations
        vector_embedding = embedding_model.encode(chunk).tolist()
        payload.append({
            "vector": vector_embedding,
            "text": chunk
        })

    if payload:
        milvus_client.insert(collection_name=COLLECTION_NAME, data=payload)
        print(f"[✓] Local Database Sync Complete: {len(payload)} chunks written to {DB_FILE_PATH}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
# =====================================================================
# 4. EXECUTION SAMPLE RUNTIME
# =====================================================================
# Upload any unstructured asset (PDF, XLSX, DOCX) to your Colab side panel files
target_sample_doc = os.path.abspath("/content/020161622x.pdf" )
#  Execute processing sequence
ingest_unstructured_document(target_sample_doc)

[*] Parsing file via MarkItDown: /content/020161622x.pdf
[*] Transforming 79 segments into vectors using BAAI/bge-large-en-v1.5...
[✓] Local Database Sync Complete: 79 chunks written to /content/local_milvus_storage.db


In [9]:
# =====================================================================
# 3. CORE EXTRACTION PIPELINE (DIRECT QUERY)
# =====================================================================

def search_local_index(query_string: str, top_k: int = 3):
    """
    Vectorizes the search question locally via BGE and executes a vector
    similarity search directly inside your local Milvus SQLite file container.
    """
    milvus_client.load_collection(collection_name=COLLECTION_NAME)
    # Math vector space translation
    query_vector = embedding_model.encode(query_string).tolist()

    # Run exact search point query
    hits = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_vector],
        limit=top_k,
        output_fields=["text"],
        search_params={"metric_type": "COSINE"}
    )

    print(f"\n🎯 Top {top_k} Results Found for: '{query_string}'")
    print("=" * 70)
    for index, hit in enumerate(hits[0]):
        print(f"\n[Match {index + 1}] Similarity Score: {hit['distance']:.4f}")
        print(f"Content Extract:\n{hit['entity']['text']}")
        print("-" * 50)

In [10]:
search_local_index("Extract structural details or metrics from data charts/tables.")


🎯 Top 3 Results Found for: 'Extract structural details or metrics from data charts/tables.'

[Match 1] Similarity Score: 0.4036
Content Extract:
|      |              |           |            | POWER         | EDITING       | 85  |
| ---- | ------------ | --------- | ---------- | ------------- | ------------- | --- |
| Name | of the class | or module | filled in  | (derived from | the filename) |     |
| Your | name and/or  | copyright | statements |               |               |     |
Skeletonsforconstructsinthatlanguage(constructoranddestruc-
| tor declarations, |     | for example) |     |     |     |     |
| ----------------- | --- | ------------ | --- | --- | --- | --- |
Another useful feature is auto-indenting. Rather than having to indent
manually (by using space or tab), the editor automatically indents for
you at the appropriate time (after typing an open brace, for example).
Thenicepartaboutthisfeatureisthatyoucanusetheeditortoprovide
project.5
| a consistent | indentation